# Train Emotion - 7 Classes (AffectNet)

Dataset da tai san tren Google Drive. Se copy sang o local Colab de train nhanh hon.

| Class | ID |
|---|---|
| Neutral | 0 |
| Happy | 1 |
| Sad | 2 |
| Surprise | 3 |
| Fear | 4 |
| Disgust | 5 |
| Anger | 6 |

**Backbone:** EfficientNetB2  
**Strategy:** 2-phase fine-tuning + class weights

In [ ]:
print('=== [1/10] Cai dat thu vien ===')
!pip -q install tensorflow pandas scikit-learn matplotlib seaborn
print('Da cai dat xong!')

In [ ]:
print('=== [2/10] Mount Google Drive ===')
from google.colab import drive
drive.mount('/content/drive')
print('Da mount Drive thanh cong!')

In [ ]:
print('=== [3/10] Copy dataset sang o local Colab (nhanh gap 5-10x) ===')
import shutil
import time
from pathlib import Path

# ====== CHINH SUA DUONG DAN O DAY ======
DRIVE_DATASET = Path('/content/drive/MyDrive/archive')
MODELS_DIR    = Path('/content/drive/MyDrive/face_age_gender_emotion/models')
# ========================================

LOCAL_DATASET = Path('/content/dataset')

MODELS_DIR.mkdir(parents=True, exist_ok=True)

if not DRIVE_DATASET.exists():
    raise FileNotFoundError(f'Khong tim thay dataset: {DRIVE_DATASET}')

# Tim thu muc chua Train/
def find_dataset_dir(root):
    for name in ['Train', 'train']:
        if (root / name).exists():
            return root
    for sub in sorted(root.iterdir()):
        if sub.is_dir():
            for name in ['Train', 'train']:
                if (sub / name).exists():
                    return sub
    raise FileNotFoundError(f'Khong tim thay Train/ trong {root}')

source_dir = find_dataset_dir(DRIVE_DATASET)
print(f'Dataset tren Drive: {source_dir}')

if LOCAL_DATASET.exists() and any(LOCAL_DATASET.rglob('*.jpg')):
    print('Dataset da co tren local, bo qua copy.')
else:
    print(f'Dang copy tu Drive sang {LOCAL_DATASET}...')
    print('(Chi can lam 1 lan, mat khoang 2-5 phut)')
    start = time.time()
    shutil.copytree(source_dir, LOCAL_DATASET, dirs_exist_ok=True)
    elapsed = time.time() - start
    print(f'Copy xong trong {elapsed:.0f}s!')

DATASET_DIR = LOCAL_DATASET
print(f'\nSe train tu: {DATASET_DIR} (local SSD, nhanh!)')

In [ ]:
print('=== [4/10] Cau hinh ===')

IMAGE_SIZE = 224
BATCH_SIZE = 64
EPOCHS_PHASE1 = 8
EPOCHS_PHASE2 = 30
MAX_TRAIN_SAMPLES = None

print(f'DATASET_DIR: {DATASET_DIR}')
print(f'MODELS_DIR:  {MODELS_DIR}')
print(f'IMAGE_SIZE={IMAGE_SIZE}, BATCH_SIZE={BATCH_SIZE}')
print(f'Phase 1: {EPOCHS_PHASE1} epochs, Phase 2: {EPOCHS_PHASE2} epochs')

print(f'\nCau truc dataset:')
for item in sorted(DATASET_DIR.iterdir()):
    if item.is_dir():
        print(f'  {item.name}/')
        for sub in sorted(item.iterdir()):
            if sub.is_dir():
                file_count = sum(1 for f in sub.iterdir() if f.is_file())
                print(f'    {sub.name}/  ({file_count} files)')
    else:
        size_mb = item.stat().st_size / (1024*1024)
        print(f'  {item.name}  ({size_mb:.1f} MB)')
print('OK!')

In [ ]:
print('=== [5/10] Doc data (7 classes) ===')
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

EMOTION_LABELS = ['Neutral', 'Happy', 'Sad', 'Surprise', 'Fear', 'Disgust', 'Anger']
NUM_CLASSES = len(EMOTION_LABELS)

FOLDER_TO_ID = {
    'neutral': 0, 'neutrality': 0, '0': 0,
    'happy': 1, 'happiness': 1, '1': 1,
    'sad': 2, 'sadness': 2, '2': 2,
    'surprise': 3, 'surprised': 3, '3': 3,
    'fear': 4, 'fearful': 4, '4': 4,
    'disgust': 5, 'disgusted': 5, '5': 5,
    'anger': 6, 'angry': 6, '6': 6,
}
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def dataframe_from_directory(root):
    rows = []
    print(f'Scan: {root}')
    for class_dir in sorted(root.iterdir()):
        if not class_dir.is_dir():
            continue
        label = FOLDER_TO_ID.get(class_dir.name.strip().lower())
        if label is None:
            print(f'  Bo qua: {class_dir.name}')
            continue
        count = 0
        for img in class_dir.rglob('*'):
            if img.is_file() and img.suffix.lower() in IMAGE_SUFFIXES:
                rows.append({'image_path': str(img), 'emotion_id': label})
                count += 1
        print(f'  {class_dir.name:>12s} -> {EMOTION_LABELS[label]:>10s} (id={label}): {count} anh')
    return pd.DataFrame(rows)

# Tim Train va Test
train_root, val_root = None, None
for name in ['Train', 'train', 'training']:
    if (DATASET_DIR / name).exists():
        train_root = DATASET_DIR / name
        break
for name in ['Test', 'test', 'Val', 'val']:
    if (DATASET_DIR / name).exists():
        val_root = DATASET_DIR / name
        break

print(f'\nTrain: {train_root}')
print(f'Val:   {val_root}')

print(f'\n--- Train ---')
train_data = dataframe_from_directory(train_root)
if val_root:
    print(f'\n--- Val/Test ---')
    val_data = dataframe_from_directory(val_root)
else:
    val_data = None

if train_data.empty:
    raise RuntimeError('Khong co anh nao!')

if val_data is None or val_data.empty:
    print('\nChia train -> train/val (85/15)...')
    train_data, val_data = train_test_split(
        train_data, test_size=0.15, random_state=42, stratify=train_data['emotion_id']
    )

if MAX_TRAIN_SAMPLES:
    train_data = train_data.sample(MAX_TRAIN_SAMPLES, random_state=42)

unique_classes = np.sort(np.unique(train_data['emotion_id'].values))
emotion_class_weights = compute_class_weight(
    'balanced', classes=unique_classes, y=train_data['emotion_id'].values
)
emotion_weight_dict = dict(zip(unique_classes.astype(int), emotion_class_weights))

print(f'\nTrain: {len(train_data)} | Val: {len(val_data)}')
for idx, count in train_data['emotion_id'].value_counts().sort_index().items():
    label = EMOTION_LABELS[idx] if idx < NUM_CLASSES else f'?({idx})'
    print(f'  {label:>10s}: {count:>6d} | weight={emotion_weight_dict.get(idx, 0):.3f}')

missing = [EMOTION_LABELS[i] for i in range(NUM_CLASSES) if i not in unique_classes]
if missing:
    print(f'⚠ THIEU: {missing}')
print('Xong!')

In [ ]:
print('=== [6/10] Tao tf.data pipeline ===')
AUTOTUNE = tf.data.AUTOTUNE

# Enable mixed precision de tang toc tren GPU
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print(f'Mixed precision: {tf.keras.mixed_precision.global_policy().name}')

def load_image(path, emotion_id, training=False):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    if training:
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, 0.15)
        image = tf.image.random_contrast(image, 0.80, 1.20)
        image = tf.image.random_saturation(image, 0.80, 1.20)
        image = tf.image.random_hue(image, 0.05)
        image = tf.clip_by_value(image, 0.0, 1.0)
    return image, emotion_id

def make_dataset(df, training):
    ds = tf.data.Dataset.from_tensor_slices((
        df['image_path'].values.astype(str),
        df['emotion_id'].values.astype('int32')
    ))
    if training:
        ds = ds.shuffle(min(len(df), 10000), seed=42, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: load_image(p, y, training), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_data, training=True)
val_ds = make_dataset(val_data, training=False)

print(f'Train batches: {tf.data.experimental.cardinality(train_ds).numpy()}')
print(f'Val batches:   {tf.data.experimental.cardinality(val_ds).numpy()}')
print('Pipeline san sang!')

In [ ]:
print('=== [7/10] Phase 1: Freeze backbone, train head ===')
from tensorflow import keras
from tensorflow.keras import layers

inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name='image')
base = keras.applications.EfficientNetB2(
    include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), weights='imagenet'
)
base.trainable = False
print(f'Backbone: EfficientNetB2 (FROZEN, {len(base.layers)} layers)')

x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D(name='gap')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.35)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.25)(x)
# float32 output cho mixed precision
x = layers.Dense(NUM_CLASSES, name='emotion_logits')(x)
outputs = layers.Activation('softmax', dtype='float32', name='emotion')(x)

model = keras.Model(inputs=inputs, outputs=outputs, name='emotion_model')
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'],
)

trainable = sum(p.numpy().size for p in model.trainable_weights)
total = sum(p.numpy().size for p in model.weights)
print(f'Trainable: {trainable:,} / {total:,}')
print(f'Training {EPOCHS_PHASE1} epochs, lr=1e-3...')

history1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_PHASE1, class_weight=emotion_weight_dict,
)
print(f'\nPhase 1 xong! Val acc: {history1.history["val_accuracy"][-1]:.4f}')

In [ ]:
print('=== [8/10] Phase 2: Unfreeze backbone, fine-tune ===')
base.trainable = True
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'],
)
print(f'Backbone UNFROZEN | Trainable: {sum(p.numpy().size for p in model.trainable_weights):,}')
print(f'Training {EPOCHS_PHASE2} epochs, lr=3e-5...')

save_path = str(MODELS_DIR / 'emotion_model.keras')
callbacks = [
    keras.callbacks.ModelCheckpoint(filepath=save_path, monitor='val_accuracy', mode='max', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=6, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
]

history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_PHASE2, callbacks=callbacks, class_weight=emotion_weight_dict,
)
print(f'\nPhase 2 xong! Best val acc: {max(history2.history["val_accuracy"]):.4f}')
print(f'Model luu tai: {save_path}')

In [ ]:
print('=== [9/10] Evaluation ===')
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

full_history = {}
for key in history1.history:
    full_history[key] = history1.history[key] + history2.history.get(key, [])
with open(MODELS_DIR / 'emotion_history.json', 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in full_history.items()}, f)
print('Da luu training history.')

print('Dang predict validation set...')
y_true, y_pred_list = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred_list.extend(np.argmax(preds, axis=1))

present_labels = sorted(set(y_true) | set(y_pred_list))
present_names = [EMOTION_LABELS[i] for i in present_labels if i < NUM_CLASSES]

print(f'\n{"="*50}')
print('CLASSIFICATION REPORT')
print(f'{"="*50}')
print(classification_report(y_true, y_pred_list, labels=present_labels, target_names=present_names))

cm = confusion_matrix(y_true, y_pred_list, labels=present_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=present_names, yticklabels=present_names, cmap='Blues')
plt.title('Emotion Confusion Matrix', fontsize=14)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.savefig(str(MODELS_DIR / 'emotion_confusion.png'), dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(full_history['accuracy'], label='Train')
axes[0].plot(full_history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[0].axvline(x=EPOCHS_PHASE1-1, color='r', linestyle='--', alpha=0.5)
axes[1].plot(full_history['loss'], label='Train')
axes[1].plot(full_history['val_loss'], label='Val')
axes[1].set_title('Loss'); axes[1].legend()
axes[1].axvline(x=EPOCHS_PHASE1-1, color='r', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(str(MODELS_DIR / 'emotion_training_curves.png'), dpi=150)
plt.show()
print('Da luu confusion matrix va training curves.')

In [ ]:
print('=== [10/10] Kiem tra model ===')
save_path = MODELS_DIR / 'emotion_model.keras'
if save_path.exists():
    size_mb = save_path.stat().st_size / (1024 * 1024)
    print(f'Model: {save_path}')
    print(f'Size:  {size_mb:.1f} MB')
    print(f'\nCac file trong models/:')
    for f in sorted(MODELS_DIR.iterdir()):
        if not f.name.startswith('.'):
            print(f'  {f.name:40s} {f.stat().st_size/(1024*1024):.1f} MB')
    print(f'\nHOAN TAT! Copy emotion_model.keras ve may local vao thu muc models/')
else:
    print('LOI: Khong tim thay model!')